# Overview
Clustering patient data provides a powerful framework for uncovering latent structure in heart-transplant survival that may be obscured in traditional modeling approaches. By segmenting patients based on pre-transplant characteristics—including demographics, clinical history, hemodynamics, comorbidities, and laboratory values—we aim to identify clinically meaningful subgroups that differ in risk profiles, physiological presentation, and post-transplant outcomes.

While clustering is inherently exploratory rather than predictive, it plays a critical role in revealing population heterogeneity and identifying candidate phenotypes that may not be well captured through standard regression-based methods. This is particularly relevant when examining gender-specific survival patterns, where complex interactions among biological, clinical, and social factors may give rise to distinct patient subpopulations that are difficult to model parametrically.

In this study, we leverage data from the United Network for Organ Sharing (UNOS) covering the most recent 10-year period to ensure temporal consistency across clinical practice, allocation policies, and data completeness. This constrained time horizon reduces confounding from long-term systemic changes and enables more interpretable, era-specific characterization of both recipient and donor profiles. By jointly analyzing these features, we aim to better understand how patient-donor interactions, including gender related differences, contribute to variation in post-transplant survival outcomes.

In [1]:
# path to user functions
import sys  
sys.path.append('../Src/')

import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from scipy import stats
import sys
from importlib.metadata import version

# initializing variables
SEED = 1776

# python modules
import utilities as u

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Scipy": sys.modules['scipy'].__version__,
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# adjust pandas display options to max
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
# adjust pandas display options to ensure full display of content
pd.set_option('display.max_colwidth', None)

  Library  Version
0  Python  3.11.13
1  Pandas    2.3.1
2   NumPy    2.3.2
3   Scipy   1.16.1


## Import Data

In [2]:
# import discretized data
df = pd.read_pickle("../Data/Heart_main.pkl")
df_dict =  pd.read_pickle("../Data/Dict_main.pkl")

In [3]:
df.head()

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,INUTERO,Gender_CAN,BloodGroup_CAN,Weight_kg_Registrsation_CAN,Height_cm_Registration_CAN,BMI_Listing_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportInhaled_CAN,InotropesIVRegistration_CAN,LifeSupportRegistration_PGE_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceTypeRegistration_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,PrimaryPaymentRegistration_CAN,DiagnosisAtListing_CAN,DiabetesType_CAN,DialysisTypeRegistration_CAN,CerebroVascularDisease_CAN,PreviousMalignancy_CAN,CreatinineRegistration_CAN,TotalSerumAlbuminRegistration_CAN,DefibrillatorImplantRegistration_CAN,HemodynamicsRegistration_SYS_CAN,HemodynamicsRegistration_PA_DIA_CAN,HemodynamicsRegistration_PA_MN_CAN,HemodynamicsRegistration_PCW_CAN,HemodynamicsRegistration_CO_CAN,InotropesVasodilatorsRegistration_SYS_CAN,InotropesVasodilatorsRegistration_DIA_CAN,InotropesVasodilatorsRegistration_MN_CAN,InotropesVasodilatorsRegistration_PCW_CAN,InotropesVasodilatorsRegistration_CO_CAN,CigaretteUse_CAN,CigaretteAbstinence_CAN,PriorCardiacSurgery_CAN,PriorCardiacSurgeryType_CAN,PriorCardiacSurgeryTypeText_CAN,DAYS_STAT1,StatusDays_1A,StatusDays_2,StatusDays_1B,StatusDays_A4,StatusDays_A5,StatusDays_A2,StatusDays_A3,StatusDays_1,StatusDays_A6,LastInactiveStatusReason,InitialWaitingListStatusCode_CAN,ReasonRemovalWaitingList_CAN,ReceivedDeceasedDonorTramsplant_CAN,TotalDayWaitList_CAN,StatusAtTransplant_CAN,Age_Listing_CAN,LifeSupportRegistration_CAN,AllocationBeginDate_CAN,RemovalWaitListDate_CAN,InitialWaitListDate_CAN,Hispanic_CAN,Ethnicity_CAN,Height_cm_Listing_CAN,Weight_kg_Listing_CAN,BMI_Listing_CALC_CAN,Height_cm_Removal_CAN,Weight_kg_Removal_CAN,BMI_Removal_CAN,COMPOSITE_DEATH_DATE,VentilatorRegistration_CAN,TransplantRegion_CAN,ValidationDateTCR_CAN,WorkIncomeRegistration_CAN,AntigenBW4_CAN,AntigenBW6_CAN,AntigenC1_CAN,AntigenC2_CAN,AntigenDR51_CAN,AntigenDR51_2_CAN,AntigenDR52_CAN,AntigenDR52_2_CAN,AntigenDR53_CAN,AntigenDR53_2_CAN,AntigenDQ1_CAN,AntigenDQ2_CAN,FunctionalStatusTransplant_CAN,MedicalConditionTransplant_CAN,STATUS_TRR,AdmissionDate_CAN,PrimaryPaymentTransplant_CAN,LifeSupportTransplant_ECMO_CAN,ResidencyStateTransplant_CAN,WorkIncomeTransplant_CAN,LifeSupportTransplant_PGE_CAN,CreatinineTransplant_CAN,DialysisBetweenRegistrationTransplant_CAN,HemodynamicsTransplant_CO_CAN,HemodynamicsTransplant_PA_DIA_CAN,HemodynamicsTransplant_PA_MN_CAN,HemodynamicsTransplant_PCW_CAN,HemodynamicsTransplant_SYS_CAN,LifeSupportTransplant_IABP_CAN,InfectionTherapyIV_CAN,InotropesIVTransplant_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_MN_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,LifeSupportMechanismTransplant_OTHER_CAN,PriorLungSurgeryAfterRegistration_CAN,AirwayDehiscencePostTransplant,AcuteRejectionEpisode,StrokePostTransplant,DialysisPostDischarge,PacemakerPostTransplant,SteroidsUse_CAN,TotalBilirubinTransplant_CAN,TransfusionAfterRegistration_CAN,VentricularDeviceTypeTransplant_CAN,VentricularDeviceBrandTransplant_CAN,VentilatorySupport_CAN,VentilatorTransplant_CAN,LifeSupportInhaledTransplant_CAN,PriorCardiacSurgeryTypeListAndTransplant_CAN,Hepatitis_B_CoreAntibody_CAN,SurfaceAntigenHEP_B_CAN,SurfaceHBVAntibodyTotalTransplant_CAN,CMVStatus_Transplant_CAN,HIV_SeroStatusTransplant_CAN,HEP_C_SerostatusStatus_CAN,EpsteinBarrSeroStatusTransplant_CAN,HIV_NAT_PreTransplant_CAN,HCV_NAT_PreTranspant_CAN,HBV_NAT_Result_CAN,COD,GraftFailStatus,GraftLifeSpanDay,LastFollowupNumber,TransplantStatus,TransplantSurvivalDay,RecipientStatus,FunctionalStatusFollowUp,TXHRT,TransplantProcedure_CAN,STATUS_TCR,LifeSupportInhaledRegistration_CAN,DeceasedRetyped_DON,CrossMatchDone,CPRA_Recent_CAN,CPRA_Peak_CAN,RejectionTreatmentWithinOneYear,PreviousTransplantSameOrgan_CAN,PreviousT

In [4]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30725 entries, 0 to 30724
Data columns (total 295 columns):
 #    Column                                        Non-Null Count  Dtype         
---   ------                                        --------------  -----         
 0    PreviousTransplantNumber_CAN                  30725 non-null  category      
 1    WaitListDiagnosisCode_CAN                     30725 non-null  category      
 2    INUTERO                                       6319 non-null   object        
 3    Gender_CAN                                    30725 non-null  category      
 4    BloodGroup_CAN                                30725 non-null  category      
 5    Weight_kg_Registrsation_CAN                   30673 non-null  float64       
 6    Height_cm_Registration_CAN                    30608 non-null  float64       
 7    BMI_Listing_CAN                               30606 non-null  float64       
 8    Citizenship_CAN                               30725 no

In [5]:
df_dict.info(max_cols=df_dict.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Feature            306 non-null    object        
 1   Description        306 non-null    object        
 2   Form               304 non-null    object        
 3   FeatureStartDate   244 non-null    datetime64[ns]
 4   FeatureEndDate     9 non-null      datetime64[ns]
 5   FormSection        306 non-null    object        
 6   DataType           306 non-null    object        
 7   SASAnalysisFormat  306 non-null    object        
 8   Comment            306 non-null    object        
 9   OrginalFeature     306 non-null    object        
 10  FeatureType        306 non-null    object        
 11  Information        306 non-null    object        
dtypes: datetime64[ns](2), object(10)
memory usage: 28.8+ KB


## Data Wrangle (Complete Data)

#### User Function(s)

In [6]:
def get_cols_by_cardinality(data, cat, dropna=True, flag=False):
    if flag:
        # return columns with cardinality > cat
        return [
            col for col in data.columns
            if data[col].nunique(dropna=dropna) > cat
        ]
    else:
        # return columns with cardinality <= cat
        return [
            col for col in data.columns
            if data[col].nunique(dropna=dropna) <= cat
        ]


def get_column_summary(data, cat=2, flag=True, dropna=True, ignore_list=None):
    """
    Categorizes columns based on unique value counts.
    
    Args:
        data: DataFrame to analyze.
        cat: The threshold for the number of unique values.
        flag: If True, finds columns > cat. If False, finds columns <= cat.
        dropna: Whether to count NaNs as a unique value.
        ignore_list: List of column names to exclude from the result.
    """
    if ignore_list is None:
        ignore_list = []

    # Identify columns based on the 'cat' threshold
    if flag:
        cols = [col for col in data.columns if data[col].nunique(dropna=dropna) > cat]
    else:
        cols = [col for col in data.columns if data[col].nunique(dropna=dropna) <= cat]

    # Efficiently remove ignored columns
    cols = [col for col in cols if col not in ignore_list]
    
    # Create a dictionary of unique values
    summary = {col: data[col].unique().tolist() for col in cols}
    
    print(f"--- Found {len(cols)} Columns (Threshold: {'>' if flag else '<='} {cat}) ---")
    
    # Truncate long lists for cleaner printing
    for key, value in summary.items():
        val_str = f"{value[:5]}..." if len(value) > 5 else f"{value}"
        print(f"{key} : {val_str}")

    return list(summary.keys())

In [7]:
# remove cols
remove_cols = ['DAYS_STAT1',
 'StatusDays_1A',
 'StatusDays_1B',
 'StatusDays_2',
 'StatusDays_1',
 'StatusDays_A2',
 'StatusDays_A3',
 'StatusDays_A4',
 'StatusDays_A5',
 'StatusDays_A6',
 'LastInactiveStatusReason',
 'ReasonRemovalWaitingList_CAN',
 'STATUS_TRR',
 'STATUS_TCR',
 'STATUS_DDR',
 'AcuteRejectionEpisode',
 'AirwayDehiscencePostTransplant',
 'StrokePostTransplant',
 'PacemakerPostTransplant',
 'DialysisPostDischarge',
 'GraftFailStatus',
 'GraftLifeSpanDay',
 'LastFollowupNumber',
 'GraftStatus',
 'TransplantStatus',
 'RecipientStatus',
 'RejectionTreatmentWithinOneYear',
 'FunctionalStatusFollowUp',
 'LengthOfStay']
# remove unwanted features
df = df.drop(columns=remove_cols).copy()

In [8]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30725 entries, 0 to 30724
Data columns (total 266 columns):
 #    Column                                        Non-Null Count  Dtype         
---   ------                                        --------------  -----         
 0    PreviousTransplantNumber_CAN                  30725 non-null  category      
 1    WaitListDiagnosisCode_CAN                     30725 non-null  category      
 2    INUTERO                                       6319 non-null   object        
 3    Gender_CAN                                    30725 non-null  category      
 4    BloodGroup_CAN                                30725 non-null  category      
 5    Weight_kg_Registrsation_CAN                   30673 non-null  float64       
 6    Height_cm_Registration_CAN                    30608 non-null  float64       
 7    BMI_Listing_CAN                               30606 non-null  float64       
 8    Citizenship_CAN                               30725 no

* **TCR:** Transplant Candidate Registration
* **TRR:** Transplant Recipient Registration

In [9]:
u.any_nans(df)

--- Missing Values Found () (Total Rows: 30,725) ---
                                     Count Percentage
PriorCardiacSurgeryTypeText_CAN      24407   79.4369%
INUTERO                              24406   79.4337%
Class2PRA_TransplantPercentage_CAN   20895   68.0065%
Class1PRA_TransplantPercentage_CAN   20690   67.3393%
TotalSerumAlbuminRegistration_CAN    20652   67.2156%
CPRA_Peak_CAN                        15213   49.5134%
CPRA_Recent_CAN                      15201   49.4744%
OtherMedsText3_DON                   12763   41.5395%
ValidationDateTCR_CAN                 7761   25.2596%
OtherMedsText2_DON                    4300   13.9951%
HemodynamicsRegistration_PCW_CAN      2924    9.5167%
HemodynamicsTransplant_PCW_CAN        2601    8.4654%
HemodynamicsRegistration_CO_CAN       1801    5.8617%
HemodynamicsTransplant_CO_CAN         1740    5.6631%
HemodynamicsTransplant_PA_MN_CAN      1551    5.0480%
HemodynamicsRegistration_PA_MN_CAN    1451    4.7225%
HemodynamicsTransplant_PA_DIA

#### Features with > 40& Missingness & Dates REMOVED

In [10]:
# remove: include Hispanic_CAN & Date features (OtherMedsText3_DON keep since its part of two other fetures)
remove_cols = ['PriorCardiacSurgeryTypeText_CAN','INUTERO', 'Class2PRA_TransplantPercentage_CAN', 'Class1PRA_TransplantPercentage_CAN',
        'TotalSerumAlbuminRegistration_CAN', 'CPRA_Peak_CAN', 'CPRA_Recent_CAN', 'Hispanic_CAN']

# remove all dates
remove_cols.extend(df.columns[df.columns.str.contains('Date')].tolist())
# 
df = df.drop(columns=remove_cols).copy()
#
df.shape

(30725, 245)

In [11]:
df.head()

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,Gender_CAN,BloodGroup_CAN,Weight_kg_Registrsation_CAN,Height_cm_Registration_CAN,BMI_Listing_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportInhaled_CAN,InotropesIVRegistration_CAN,LifeSupportRegistration_PGE_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceTypeRegistration_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,PrimaryPaymentRegistration_CAN,DiagnosisAtListing_CAN,DiabetesType_CAN,DialysisTypeRegistration_CAN,CerebroVascularDisease_CAN,PreviousMalignancy_CAN,CreatinineRegistration_CAN,DefibrillatorImplantRegistration_CAN,HemodynamicsRegistration_SYS_CAN,HemodynamicsRegistration_PA_DIA_CAN,HemodynamicsRegistration_PA_MN_CAN,HemodynamicsRegistration_PCW_CAN,HemodynamicsRegistration_CO_CAN,InotropesVasodilatorsRegistration_SYS_CAN,InotropesVasodilatorsRegistration_DIA_CAN,InotropesVasodilatorsRegistration_MN_CAN,InotropesVasodilatorsRegistration_PCW_CAN,InotropesVasodilatorsRegistration_CO_CAN,CigaretteUse_CAN,CigaretteAbstinence_CAN,PriorCardiacSurgery_CAN,PriorCardiacSurgeryType_CAN,InitialWaitingListStatusCode_CAN,ReceivedDeceasedDonorTramsplant_CAN,TotalDayWaitList_CAN,StatusAtTransplant_CAN,Age_Listing_CAN,LifeSupportRegistration_CAN,Ethnicity_CAN,Height_cm_Listing_CAN,Weight_kg_Listing_CAN,BMI_Listing_CALC_CAN,Height_cm_Removal_CAN,Weight_kg_Removal_CAN,BMI_Removal_CAN,COMPOSITE_DEATH_DATE,VentilatorRegistration_CAN,TransplantRegion_CAN,WorkIncomeRegistration_CAN,AntigenBW4_CAN,AntigenBW6_CAN,AntigenC1_CAN,AntigenC2_CAN,AntigenDR51_CAN,AntigenDR51_2_CAN,AntigenDR52_CAN,AntigenDR52_2_CAN,AntigenDR53_CAN,AntigenDR53_2_CAN,AntigenDQ1_CAN,AntigenDQ2_CAN,FunctionalStatusTransplant_CAN,MedicalConditionTransplant_CAN,PrimaryPaymentTransplant_CAN,LifeSupportTransplant_ECMO_CAN,ResidencyStateTransplant_CAN,WorkIncomeTransplant_CAN,LifeSupportTransplant_PGE_CAN,CreatinineTransplant_CAN,DialysisBetweenRegistrationTransplant_CAN,HemodynamicsTransplant_CO_CAN,HemodynamicsTransplant_PA_DIA_CAN,HemodynamicsTransplant_PA_MN_CAN,HemodynamicsTransplant_PCW_CAN,HemodynamicsTransplant_SYS_CAN,LifeSupportTransplant_IABP_CAN,InfectionTherapyIV_CAN,InotropesIVTransplant_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_MN_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,LifeSupportMechanismTransplant_OTHER_CAN,PriorLungSurgeryAfterRegistration_CAN,SteroidsUse_CAN,TotalBilirubinTransplant_CAN,TransfusionAfterRegistration_CAN,VentricularDeviceTypeTransplant_CAN,VentricularDeviceBrandTransplant_CAN,VentilatorySupport_CAN,VentilatorTransplant_CAN,LifeSupportInhaledTransplant_CAN,PriorCardiacSurgeryTypeListAndTransplant_CAN,Hepatitis_B_CoreAntibody_CAN,SurfaceAntigenHEP_B_CAN,SurfaceHBVAntibodyTotalTransplant_CAN,CMVStatus_Transplant_CAN,HIV_SeroStatusTransplant_CAN,HEP_C_SerostatusStatus_CAN,EpsteinBarrSeroStatusTransplant_CAN,HIV_NAT_PreTransplant_CAN,HCV_NAT_PreTranspant_CAN,HBV_NAT_Result_CAN,COD,TransplantSurvivalDay,TXHRT,TransplantProcedure_CAN,LifeSupportInhaledRegistration_CAN,DeceasedRetyped_DON,CrossMatchDone,PreviousTransplantSameOrgan_CAN,PreviousTransplantAnyOrgan_CAN,AntigenDA1_DON,AntigenDA2_DON,AntigenDB1_DON,AntigenDB2_DON,AntigenDDR1_DON,AntigenDDR2_DON,AntigenRA1_CAN,AntigenRA2_CAN,AntigenRB1_CAN,AntigenRB2_CAN,AntigenRDR1_CAN,AntigenRDR2_CAN,MismatchLevel_AMIS,MismatchLevel_BMIS,MismatchLevel_DRMIS,MismatchLevel_HLAMIS,MalignancyBetweenRegistrationTransplant_CAN,CMV_IGG_Transplant_CAN,CMV_IGM_Transplant_CAN,Citizenship_DON,PastCocaineUse_DON,Age_DON,Ethnicity_DON,Hepatitis_B_CoreAntibody_DON,SurfaceAntigenHEP_B_DON,BloodGroup_DON,HeavyAlcoholUse_DON,DeceasedOrLiving_DON,Gender_DON,ResidencyState_DON,Antibody_HEP_C_DON,NonHeartBeating_DON,AntiHypertensive_DON,BloodInfectionSource_DON,BloodUreaNitrogenLevel_DON,Creatinine_DON,Othe

In [12]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30725 entries, 0 to 30724
Data columns (total 245 columns):
 #    Column                                        Non-Null Count  Dtype   
---   ------                                        --------------  -----   
 0    PreviousTransplantNumber_CAN                  30725 non-null  category
 1    WaitListDiagnosisCode_CAN                     30725 non-null  category
 2    Gender_CAN                                    30725 non-null  category
 3    BloodGroup_CAN                                30725 non-null  category
 4    Weight_kg_Registrsation_CAN                   30673 non-null  float64 
 5    Height_cm_Registration_CAN                    30608 non-null  float64 
 6    BMI_Listing_CAN                               30606 non-null  float64 
 7    Citizenship_CAN                               30725 non-null  category
 8    ResidencyStateRegistration_CAN                30725 non-null  category
 9    EducationLevel_CAN                   

In [13]:
cols = df.columns[~df.columns.str.endswith(('_DON','_CAN'))].tolist()
cols

['COMPOSITE_DEATH_DATE',
 'COD',
 'TransplantSurvivalDay',
 'TXHRT',
 'CrossMatchDone',
 'MismatchLevel_AMIS',
 'MismatchLevel_BMIS',
 'MismatchLevel_DRMIS',
 'MismatchLevel_HLAMIS',
 'BloodGroupMatchLevel']

In [14]:
df_dict[df_dict.Feature.isin(cols)]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
2,BloodGroupMatchLevel,DONOR-RECIPIENT ABO MATCH LEVEL,CALCULATED,NaT,NaT,,CHAR(1),ABOMAT,,ABO_MAT,Category,FMTNAME: ABOMAT
10,MismatchLevel_AMIS,A LOCUS MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,AMIS,Category,
18,MismatchLevel_BMIS,B LOCUS MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,BMIS,Category,
39,COD,RECIPIENT PRIMARY CAUSE OF DEATH,TRF/TRR,1987-10-01,NaT,PATIENT STATUS,NUM,ALL_COD,,COD,Category,FMTNAME: DON_COD
41,COMPOSITE_DEATH_DATE,Composite Patient Death Date from OPTN or Verified from External Sources,TRR/TRF-CALCULATED,NaT,NaT,,NUM,,,COMPOSITE_DEATH_DATE,Category,
49,CrossMatchDone,CROSSMATCH DONE Y/N,RH,1994-04-01,NaT,TEST INFORMATION,CHAR(1),,,CRSMATCH_DONE,Category,N/Y/X to No/Yes/Missing
90,MismatchLevel_DRMIS,DR Locus MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,DRMIS,Category,
155,MismatchLevel_HLAMIS,HLA MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,HLAMIS,Category,
246,TransplantSurvivalDay,Patient Survival Time in days (based on composite death date),CALCULATED,NaT,NaT,,NUM,,,PTIME,Numeric,** LABEL **
286,TXHRT,SIMULTANEOUS HEART,CALCULATED,NaT,NaT,,CHAR(1),,,TXHRT,Category,N/Y/U/X to No/Yes/Unknown/Missing


In [15]:
df[cols].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
COMPOSITE_DEATH_DATE,30725,3069,998,24367,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COD,30725,72,998,25221,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TransplantSurvivalDay,30300.0,NaN,NaN,NaN,1433.407195,1156.055867,0.0,369.0,1120.0,2207.0,4300.0
TXHRT,30725,1,Yes,30725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CrossMatchDone,30725,3,Yes,28294,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_AMIS,30725,4,2,14469,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_BMIS,30725,4,2,20235,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_DRMIS,30725,4,2,15395,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_HLAMIS,30725,8,5,10469,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BloodGroupMatchLevel,30725,3,Identical,26298,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# remove label
cols.remove('TransplantSurvivalDay')
# use for both
cols.remove('CrossMatchDone')
cols.remove('MismatchLevel_AMIS')
cols.remove('MismatchLevel_BMIS')
cols.remove('MismatchLevel_DRMIS')
cols.remove('MismatchLevel_HLAMIS')
cols.remove('BloodGroupMatchLevel')
# display
cols

['COMPOSITE_DEATH_DATE', 'COD', 'TXHRT']

In [17]:
# remove unwanted features
df = df.drop(cols, axis=1).copy()

In [18]:
# sanity check
cols = df.columns[~df.columns.str.endswith(('_DON','_CAN'))].tolist()
cols_add = cols
# remove label
cols_add.remove('TransplantSurvivalDay')
# display
cols_add

['CrossMatchDone',
 'MismatchLevel_AMIS',
 'MismatchLevel_BMIS',
 'MismatchLevel_DRMIS',
 'MismatchLevel_HLAMIS',
 'BloodGroupMatchLevel']

In [19]:
u.any_nans(df)

--- Missing Values Found () (Total Rows: 30,725) ---
                                     Count Percentage
OtherMedsText3_DON                   12763   41.5395%
OtherMedsText2_DON                    4300   13.9951%
HemodynamicsRegistration_PCW_CAN      2924    9.5167%
HemodynamicsTransplant_PCW_CAN        2601    8.4654%
HemodynamicsRegistration_CO_CAN       1801    5.8617%
HemodynamicsTransplant_CO_CAN         1740    5.6631%
HemodynamicsTransplant_PA_MN_CAN      1551    5.0480%
HemodynamicsRegistration_PA_MN_CAN    1451    4.7225%
HemodynamicsTransplant_PA_DIA_CAN     1252    4.0749%
HemodynamicsTransplant_SYS_CAN        1228    3.9967%
HemodynamicsRegistration_PA_DIA_CAN   1134    3.6908%
HemodynamicsRegistration_SYS_CAN      1112    3.6192%
OtherMedsText1_DON                     911    2.9650%
IschemicTimeHour_DON                   592    1.9268%
TotalBilirubinTransplant_CAN           484    1.5753%
TransplantSurvivalDay                  425    1.3832%
CreatinineTransplant_CAN     

In [20]:
# remove any NaNs for the label
df.dropna(subset=['TransplantSurvivalDay'], inplace=True)
# shape
df.shape

(30300, 242)

### Ordinal
* In this research, missing or unknown data often reflects something meaningful—like incomplete records, different workflows, or unmeasured risk factors. In this context, we will be treating it as a nominal category. That way, we preserve its interpretability, and our model can account for it as a distinct group, rather than risking an incorrect ordering assumption.

In [21]:
# ordinal features
ordinal_cols = ['PreviousTransplantNumber_CAN', 'EducationLevel_CAN', 'FunctionalStatusRegistration_CAN', 
                'FunctionalStatusTransplant_CAN', 'CigaretteAbstinence_CAN', 'MedicalConditionTransplant_CAN']

# display
df.loc[:, ordinal_cols].head()

,PreviousTransplantNumber_CAN,EducationLevel_CAN,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN,CigaretteAbstinence_CAN,MedicalConditionTransplant_CAN
0,0,ATTENDED COLLEGE/TECHNICAL SCHOOL,"10% - Moribund, fatal processes progressing rapidly","10% - Moribund, fatal processes progressing rapidly",Unknown,In Intensive Care Unit
1,0,HIGH SCHOOL (9-12) or GED,"10% - Moribund, fatal processes progressing rapidly",90% - Able to carry on normal activity: minor symptoms of disease,Unknown,Not Hospitalized
2,0,ATTENDED COLLEGE/TECHNICAL SCHOOL,"10% - Moribund, fatal processes progressing rapidly","20% - Very sick, hospitalization necessary: active treatment necessary",Unknown,Hospitalized Not in ICU
3,0,HIGH SCHOOL (9-12) or GED,"20% - Very sick, hospitalization necessary: active treatment necessary",80% - Normal activity with effort: some symptoms of disease,0-2 months,Not Hospitalized
4,0,ASSOCIATE/BACHELOR DEGREE,90% - Able to carry on normal activity: minor symptoms of disease,90% - Able to carry on normal activity: minor symptoms of disease,Unknown,Not Hospitalized


#### PreviousTransplantNumber_CAN

In [22]:
# info
feature = u.get_feature_info(df, 'PreviousTransplantNumber_CAN', cat=True)

                              count unique top   freq
PreviousTransplantNumber_CAN  30300      4   0  29303

:::: NaN Count:
PreviousTransplantNumber_CAN    0 

--- PreviousTransplantNumber_CAN ---
dtype: category
Categories: ['0', '1', '2', '3']
Ordered: False



#### EducationLevel_CAN

In [23]:
# info
feature = u.get_feature_info(df, 'EducationLevel_CAN', cat=True)

                    count unique                        top   freq
EducationLevel_CAN  30300      7  HIGH SCHOOL (9-12) or GED  11154

:::: NaN Count:
EducationLevel_CAN    0 

--- EducationLevel_CAN ---
dtype: category
Categories: ['ASSOCIATE/BACHELOR DEGREE', 'ATTENDED COLLEGE/TECHNICAL SCHOOL', 'GRADE SCHOOL (0-8)', 'HIGH SCHOOL (9-12) or GED', 'NONE', 'POST-COLLEGE GRADUATE DEGREE', 'Unknown']
Ordered: False



#### FunctionalStatus

In [24]:
# info
feature = u.get_feature_info(df, 'FunctionalStatusRegistration_CAN', cat=True)

                                  count unique                                                                     top  freq
FunctionalStatusRegistration_CAN  30300     24  20% - Very sick, hospitalization necessary: active treatment necessary  6797

:::: NaN Count:
FunctionalStatusRegistration_CAN    0 

--- FunctionalStatusRegistration_CAN ---
dtype: category
Categories: [996, '10% - Moribund, fatal processes progressing rapidly', '10% - No play; does not get out of bed', '100% - Fully active, normal', '100% - Normal, no complaints, no evidence of disease', '20% - Often sleeping; play entirely limited to very passive activities', '20% - Very sick, hospitalization necessary: active treatment necessary', '30% - In bed; needs assistance even for quiet play', '30% - Severely disabled: hospitalization is indicated, death not imminent', '40% - Disabled: requires special care and assistance', '40% - Mostly in bed; participates in quiet activities', '50% - Can dress but lies around much of d

In [25]:
df.FunctionalStatusRegistration_CAN.value_counts(dropna=False)

FunctionalStatusRegistration_CAN
20% - Very sick, hospitalization necessary: active treatment necessary                                 6797
70% - Cares for self: unable to carry on normal activity or active work                                4334
40% - Disabled: requires special care and assistance                                                   4122
60% - Requires occasional assistance but is able to care for needs                                     3756
50% - Requires considerable assistance and frequent medical care                                       3229
80% - Normal activity with effort: some symptoms of disease                                            2678
30% - Severely disabled: hospitalization is indicated, death not imminent                              2647
Unknown                                                                                                 973
90% - Able to carry on normal activity: minor symptoms of disease                                      

In [26]:
mask = (
    ~df['FunctionalStatusRegistration_CAN'].astype(str).str.contains("%", na=False)
    & (df['FunctionalStatusRegistration_CAN'] != "Unknown")
)

# display
df.loc[mask, ['FunctionalStatusRegistration_CAN', 'FunctionalStatusTransplant_CAN']]

,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN
3995,Performs activities of daily living with SOME assistance,80% - Normal activity with effort: some symptoms of disease
14942,Performs activities of daily living with SOME assistance,"20% - Very sick, hospitalization necessary: active treatment necessary"
24833,Performs activities of daily living with SOME assistance,60% - Requires occasional assistance but is able to care for needs
24866,996,Unknown
25170,Performs activities of daily living with NO assistance,70% - Cares for self: unable to carry on normal activity or active work
25219,Performs activities of daily living with NO assistance,40% - Disabled: requires special care and assistance
25365,Performs activities of daily living with NO assistance,"20% - Very sick, hospitalization necessary: active treatment necessary"
25947,996,"20% - Very sick, hospitalization necessary: active treatment necessary"
26069,Performs activities of daily living with NO assistance,80% - Normal activity with effort: some symptoms of disease
26134,Performs activities of daily living with SOME assistance,50% - Requires considerable assistance and frequent medical care


In [27]:
# remove these rows (Not Applicable (patient < 1 year old))
df[df["FunctionalStatusRegistration_CAN"].eq(996)]

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,Gender_CAN,BloodGroup_CAN,Weight_kg_Registrsation_CAN,Height_cm_Registration_CAN,BMI_Listing_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportInhaled_CAN,InotropesIVRegistration_CAN,LifeSupportRegistration_PGE_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceTypeRegistration_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,PrimaryPaymentRegistration_CAN,DiagnosisAtListing_CAN,DiabetesType_CAN,DialysisTypeRegistration_CAN,CerebroVascularDisease_CAN,PreviousMalignancy_CAN,CreatinineRegistration_CAN,DefibrillatorImplantRegistration_CAN,HemodynamicsRegistration_SYS_CAN,HemodynamicsRegistration_PA_DIA_CAN,HemodynamicsRegistration_PA_MN_CAN,HemodynamicsRegistration_PCW_CAN,HemodynamicsRegistration_CO_CAN,InotropesVasodilatorsRegistration_SYS_CAN,InotropesVasodilatorsRegistration_DIA_CAN,InotropesVasodilatorsRegistration_MN_CAN,InotropesVasodilatorsRegistration_PCW_CAN,InotropesVasodilatorsRegistration_CO_CAN,CigaretteUse_CAN,CigaretteAbstinence_CAN,PriorCardiacSurgery_CAN,PriorCardiacSurgeryType_CAN,InitialWaitingListStatusCode_CAN,ReceivedDeceasedDonorTramsplant_CAN,TotalDayWaitList_CAN,StatusAtTransplant_CAN,Age_Listing_CAN,LifeSupportRegistration_CAN,Ethnicity_CAN,Height_cm_Listing_CAN,Weight_kg_Listing_CAN,BMI_Listing_CALC_CAN,Height_cm_Removal_CAN,Weight_kg_Removal_CAN,BMI_Removal_CAN,VentilatorRegistration_CAN,TransplantRegion_CAN,WorkIncomeRegistration_CAN,AntigenBW4_CAN,AntigenBW6_CAN,AntigenC1_CAN,AntigenC2_CAN,AntigenDR51_CAN,AntigenDR51_2_CAN,AntigenDR52_CAN,AntigenDR52_2_CAN,AntigenDR53_CAN,AntigenDR53_2_CAN,AntigenDQ1_CAN,AntigenDQ2_CAN,FunctionalStatusTransplant_CAN,MedicalConditionTransplant_CAN,PrimaryPaymentTransplant_CAN,LifeSupportTransplant_ECMO_CAN,ResidencyStateTransplant_CAN,WorkIncomeTransplant_CAN,LifeSupportTransplant_PGE_CAN,CreatinineTransplant_CAN,DialysisBetweenRegistrationTransplant_CAN,HemodynamicsTransplant_CO_CAN,HemodynamicsTransplant_PA_DIA_CAN,HemodynamicsTransplant_PA_MN_CAN,HemodynamicsTransplant_PCW_CAN,HemodynamicsTransplant_SYS_CAN,LifeSupportTransplant_IABP_CAN,InfectionTherapyIV_CAN,InotropesIVTransplant_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_MN_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,LifeSupportMechanismTransplant_OTHER_CAN,PriorLungSurgeryAfterRegistration_CAN,SteroidsUse_CAN,TotalBilirubinTransplant_CAN,TransfusionAfterRegistration_CAN,VentricularDeviceTypeTransplant_CAN,VentricularDeviceBrandTransplant_CAN,VentilatorySupport_CAN,VentilatorTransplant_CAN,LifeSupportInhaledTransplant_CAN,PriorCardiacSurgeryTypeListAndTransplant_CAN,Hepatitis_B_CoreAntibody_CAN,SurfaceAntigenHEP_B_CAN,SurfaceHBVAntibodyTotalTransplant_CAN,CMVStatus_Transplant_CAN,HIV_SeroStatusTransplant_CAN,HEP_C_SerostatusStatus_CAN,EpsteinBarrSeroStatusTransplant_CAN,HIV_NAT_PreTransplant_CAN,HCV_NAT_PreTranspant_CAN,HBV_NAT_Result_CAN,TransplantSurvivalDay,TransplantProcedure_CAN,LifeSupportInhaledRegistration_CAN,DeceasedRetyped_DON,CrossMatchDone,PreviousTransplantSameOrgan_CAN,PreviousTransplantAnyOrgan_CAN,AntigenDA1_DON,AntigenDA2_DON,AntigenDB1_DON,AntigenDB2_DON,AntigenDDR1_DON,AntigenDDR2_DON,AntigenRA1_CAN,AntigenRA2_CAN,AntigenRB1_CAN,AntigenRB2_CAN,AntigenRDR1_CAN,AntigenRDR2_CAN,MismatchLevel_AMIS,MismatchLevel_BMIS,MismatchLevel_DRMIS,MismatchLevel_HLAMIS,MalignancyBetweenRegistrationTransplant_CAN,CMV_IGG_Transplant_CAN,CMV_IGM_Transplant_CAN,Citizenship_DON,PastCocaineUse_DON,Age_DON,Ethnicity_DON,Hepatitis_B_CoreAntibody_DON,SurfaceAntigenHEP_B_DON,BloodGroup_DON,HeavyAlcoholUse_DON,DeceasedOrLiving_DON,Gender_DON,ResidencyState_DON,Antibody_HEP_C_DON,NonHeartBeating_DON,AntiHypertensive_DON,BloodInfectionSource_DON,BloodUreaNitrogenLevel_DON,Creatinine_DON,OtherInfectionSource_DON,Diuretics_

In [28]:
# remove these two rows
df = df[df["FunctionalStatusRegistration_CAN"].ne(996)].reset_index(drop=True).copy()

In [29]:
mask = (
    ~df['FunctionalStatusRegistration_CAN'].astype(str).str.contains("%", na=False)
    & (df['FunctionalStatusRegistration_CAN'] != "Unknown")
)

# display
df.loc[mask, ['WorkIncomeRegistration_CAN', 'FunctionalStatusRegistration_CAN', 'FunctionalStatusTransplant_CAN']]

,WorkIncomeRegistration_CAN,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN
3995,No,Performs activities of daily living with SOME assistance,80% - Normal activity with effort: some symptoms of disease
14942,No,Performs activities of daily living with SOME assistance,"20% - Very sick, hospitalization necessary: active treatment necessary"
24408,Unknown,Performs activities of daily living with SOME assistance,60% - Requires occasional assistance but is able to care for needs
24744,No,Performs activities of daily living with NO assistance,70% - Cares for self: unable to carry on normal activity or active work
24793,Unknown,Performs activities of daily living with NO assistance,40% - Disabled: requires special care and assistance
24939,Unknown,Performs activities of daily living with NO assistance,"20% - Very sick, hospitalization necessary: active treatment necessary"
25642,Yes,Performs activities of daily living with NO assistance,80% - Normal activity with effort: some symptoms of disease
25707,Unknown,Performs activities of daily living with SOME assistance,50% - Requires considerable assistance and frequent medical care
25741,Unknown,Performs activities of daily living with NO assistance,60% - Requires occasional assistance but is able to care for needs
25752,Unknown,Performs activities of daily living with SOME assistance,60% - Requires occasional assistance but is able to care for needs


In [30]:
# mapping
mapping = {'Performs activities of daily living with SOME assistance': '60% - Performs activities of daily living with SOME assistance',
           'Performs activities of daily living with NO assistance': '70% - Performs activities of daily living with NO assistance'}

# apply mapping
df['FunctionalStatusRegistration_CAN'] = df['FunctionalStatusRegistration_CAN'].map(mapping).fillna(df['FunctionalStatusRegistration_CAN'])

In [31]:
# display
df.loc[mask, ['WorkIncomeRegistration_CAN', 'FunctionalStatusRegistration_CAN', 'FunctionalStatusTransplant_CAN']]

,WorkIncomeRegistration_CAN,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN
3995,No,60% - Performs activities of daily living with SOME assistance,80% - Normal activity with effort: some symptoms of disease
14942,No,60% - Performs activities of daily living with SOME assistance,"20% - Very sick, hospitalization necessary: active treatment necessary"
24408,Unknown,60% - Performs activities of daily living with SOME assistance,60% - Requires occasional assistance but is able to care for needs
24744,No,70% - Performs activities of daily living with NO assistance,70% - Cares for self: unable to carry on normal activity or active work
24793,Unknown,70% - Performs activities of daily living with NO assistance,40% - Disabled: requires special care and assistance
24939,Unknown,70% - Performs activities of daily living with NO assistance,"20% - Very sick, hospitalization necessary: active treatment necessary"
25642,Yes,70% - Performs activities of daily living with NO assistance,80% - Normal activity with effort: some symptoms of disease
25707,Unknown,60% - Performs activities of daily living with SOME assistance,50% - Requires considerable assistance and frequent medical care
25741,Unknown,70% - Performs activities of daily living with NO assistance,60% - Requires occasional assistance but is able to care for needs
25752,Unknown,60% - Performs activities of daily living with SOME assistance,60% - Requires occasional assistance but is able to care for needs


In [32]:
# regex: 
# Ignore any leading spaces: \s*
# Capture the number just before %: (\d+(?:\.\d+)?)
# Ignore spaces before the %: \s*
# Look ahead to ensure the next character is %  (but don’t consume it): (?=%)
df["FunctionalStatusRegistration_CAN"] = (
    df["FunctionalStatusRegistration_CAN"]
    .str.extract(r"\s*(\d+(?:\.\d+)?)\s*(?=%)", expand=False)
    .astype('category')
)
df["FunctionalStatusTransplant_CAN"] = (
    df["FunctionalStatusTransplant_CAN"]
    .str.extract(r"\s*(\d+(?:\.\d+)?)\s*(?=%)", expand=False)
    .astype('category')
)
#
features = u.get_feature_info(df, 'FunctionalStatus', cat=True)

                                  count unique top  freq
FunctionalStatusRegistration_CAN  29325     10  20  6801
FunctionalStatusTransplant_CAN    28924     10  20  8611

:::: NaN Count:
FunctionalStatusRegistration_CAN     973
FunctionalStatusTransplant_CAN      1374 

--- FunctionalStatusRegistration_CAN ---
dtype: category
Categories: ['10', '100', '20', '30', '40', '50', '60', '70', '80', '90']
Ordered: False

--- FunctionalStatusTransplant_CAN ---
dtype: category
Categories: ['10', '100', '20', '30', '40', '50', '60', '70', '80', '90']
Ordered: False



In [33]:
# verify Missing is Random
u.check_informative_missingness(df, 'FunctionalStatusRegistration_CAN', target='TransplantSurvivalDay', unknown_val=None)

--- Missingness (): FunctionalStatusRegistration_CAN ---
Group Sizes:    (Unknown=973, Known=29,325)
Mean survival:  (Unknown=1,184.8d, Known=1,441.5d)
Difference:     -256.7 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0000
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -0.2222 (Small)
95% CI:        [-0.2861, -0.1583]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



In [34]:
# verify Missing is Random
u.check_informative_missingness(df, 'FunctionalStatusTransplant_CAN', target='TransplantSurvivalDay', unknown_val=None)

--- Missingness (): FunctionalStatusTransplant_CAN ---
Group Sizes:    (Unknown=1,374, Known=28,924)
Mean survival:  (Unknown=1,129.3d, Known=1,447.7d)
Difference:     -318.4 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0000
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -0.2759 (Small)
95% CI:        [-0.3300, -0.2217]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



In [35]:
# display
df.loc[:,['WorkIncomeRegistration_CAN', 'FunctionalStatusRegistration_CAN', 'FunctionalStatusTransplant_CAN']].sample(n=5, random_state=SEED)

,WorkIncomeRegistration_CAN,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN
24713,No,30,20
12770,No,60,50
12524,No,10,20
14592,No,30,60
15126,No,80,NaN


#### CigaretteAbstinence_CAN

In [36]:
# info
feature = u.get_feature_info(df, 'CigaretteAbstinence_CAN', cat=True)

                         count unique      top   freq
CigaretteAbstinence_CAN  30298     10  Unknown  16933

:::: NaN Count:
CigaretteAbstinence_CAN    0 

--- CigaretteAbstinence_CAN ---
dtype: category
Categories: ['0-2 months', '13-24 months', '25-36 months', '3-12 months', '37-48 months', '49-60 months', '>60 months', 'Continues to smoke', 'Unknown', 'Unknown duration']
Ordered: False



In [37]:
# verify Missing is Random
u.check_informative_missingness(df, 'CigaretteAbstinence_CAN', target='TransplantSurvivalDay', unknown_val='Unknown')

--- Missingness (): CigaretteAbstinence_CAN ---
Group Sizes:    (Unknown=16,933, Known=13,365)
Mean survival:  (Unknown=1,420.2d, Known=1,449.7d)
Difference:     -29.5 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0270
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -0.0256 (Negligible)
95% CI:        [-0.0482, -0.0029]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



#### MedicalConditionTransplant_CAN

In [38]:
# info
feature = u.get_feature_info(df, 'MedicalConditionTransplant_CAN', cat=True)

                                count unique               top   freq
MedicalConditionTransplant_CAN  30298      4  Not Hospitalized  14506

:::: NaN Count:
MedicalConditionTransplant_CAN    0 

--- MedicalConditionTransplant_CAN ---
dtype: category
Categories: ['Hospitalized Not in ICU', 'In Intensive Care Unit', 'Not Hospitalized', 'Unknown']
Ordered: False



In [39]:
# verify Missing is Random
u.check_informative_missingness(df, feature[0], target='TransplantSurvivalDay', unknown_val='Unknown')

--- Missingness (): MedicalConditionTransplant_CAN ---
Group Sizes:    (Unknown=45, Known=30,253)
Mean survival:  (Unknown=102.4d, Known=1,435.2d)
Difference:     -1,332.8 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0000
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -1.1542 (Large)
95% CI:        [-1.4467, -0.8616]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



### Binary Columns

In [40]:
# display category features
_ = get_column_summary(df, cat = 2, flag=False, ignore_list=None)

--- Found 22 Columns (Threshold: <= 2) ---
Gender_CAN : ['M', 'F']
LifeSupportRegistration_ECMO_CAN : ['No', 'Yes']
LifeSupportRegistration_IABP_CAN : ['No', 'Yes']
LifeSupportInhaled_CAN : ['No', 'Yes']
InotropesIVRegistration_CAN : ['No', 'Yes']
LifeSupportRegistration_PGE_CAN : ['No', 'Yes']
LifeSupportMechanismRegistration_OTHER_CAN : ['Yes', 'No']
ReceivedDeceasedDonorTramsplant_CAN : ['Yes', 'No']
VentilatorRegistration_CAN : ['No', 'Yes']
LifeSupportTransplant_ECMO_CAN : ['No', 'Yes']
LifeSupportTransplant_PGE_CAN : ['No', 'Yes']
LifeSupportTransplant_IABP_CAN : ['No', 'Yes']
InotropesIVTransplant_CAN : ['No', 'Yes']
LifeSupportMechanismTransplant_OTHER_CAN : ['Yes', 'No']
VentilatorTransplant_CAN : ['No', 'Yes']
LifeSupportInhaledTransplant_CAN : ['No', 'Yes']
TransplantProcedure_CAN : ['501']
LifeSupportInhaledRegistration_CAN : ['No', 'Yes']
PreviousTransplantSameOrgan_CAN : ['No', 'Yes']
PreviousTransplantAnyOrgan_CAN : ['No', 'Yes']
DeceasedOrLiving_DON : ['Deceased Donor',

In [41]:
df.TransplantProcedure_CAN.value_counts()

TransplantProcedure_CAN
501    30298
Name: count, dtype: int64

In [42]:
# add to remove cols list
remove_cols = ['TransplantProcedure_CAN']

In [43]:
# ignore for next run
ignore_cols = get_cols_by_cardinality(df, 2, dropna=True, flag=False)
# display category features
cols = get_column_summary(df, cat = 3, flag=False, ignore_list=ignore_cols)

--- Found 74 Columns (Threshold: <= 3) ---
CerebroVascularDisease_CAN : ['No', 'Yes', 'Unknown']
PreviousMalignancy_CAN : ['No', 'Yes', 'Unknown']
DefibrillatorImplantRegistration_CAN : ['Yes', 'No', 'Unknown']
InotropesVasodilatorsRegistration_SYS_CAN : ['Yes', 'No', 'Unknown']
InotropesVasodilatorsRegistration_DIA_CAN : ['Yes', 'No', 'Unknown']
InotropesVasodilatorsRegistration_MN_CAN : ['Yes', 'No', 'Unknown']
InotropesVasodilatorsRegistration_PCW_CAN : ['Yes', 'No', 'Unknown']
InotropesVasodilatorsRegistration_CO_CAN : ['Yes', 'No', 'Unknown']
CigaretteUse_CAN : ['No', 'Yes', 'Unknown']
PriorCardiacSurgery_CAN : ['Yes', 'No', 'Unknown']
LifeSupportRegistration_CAN : ['Yes', 'No', 'Unknown']
WorkIncomeRegistration_CAN : ['No', 'Yes', 'Unknown']
WorkIncomeTransplant_CAN : ['No', 'Unknown', 'Yes']
DialysisBetweenRegistrationTransplant_CAN : ['No', 'Yes', 'Unknown']
InfectionTherapyIV_CAN : ['No', 'Unknown', 'Yes']
InotropesVasodilatorsTransplant_CO_CAN : ['Unknown', 'No', 'Yes']
Inotr

In [44]:
# verify Missing is Random
u.check_informative_missingness(df, 'TransplantType_CAN', target='TransplantSurvivalDay', unknown_val='Unknown')

--- Missingness (): TransplantType_CAN ---
Group Sizes:    (Unknown=54, Known=30,244)
Mean survival:  (Unknown=99.6d, Known=1,435.6d)
Difference:     -1,336.0 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0000
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -1.1571 (Large)
95% CI:        [-1.4243, -0.8900]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



In [45]:
# no value added
df = df.drop(remove_cols, axis=1).copy()

### Candidate

In [46]:
# keep candidate and TransplantSurvivalDay features
cols = df.columns[df.columns.str.contains("CAN")].to_list()
cols.extend(['TransplantSurvivalDay'])
cols.extend(cols_add)

# all the candidates' features and TransplantSurvivalDay
df_can = df[cols]

# display
df_can.shape

(30298, 146)

### Donor

In [47]:
# keep donor and TransplantSurvivalDay features
cols = df.columns[df.columns.str.contains("DON")].to_list()
cols.extend(['TransplantSurvivalDay'])
cols.extend(cols_add)

# all the candidates' features and TransplantSurvivalDay
df_don = df[cols]

# display
df_don.shape

(30298, 102)

In [48]:
# sanity check
df.shape[1], df_can.shape[1] + df_don.shape[1]

(241, 248)

#### Write to Disk: Full Data

In [49]:
# save data: heart dataset
u.write_to_file(df_don, 'DON_Heart_Data',path='../Data/', format='pkl')
u.write_to_file(df_can, 'CAN_Heart_Data',path='../Data/', format='pkl')

30,298 records written to ../Data/DON_Heart_Data.pkl
30,298 records written to ../Data/CAN_Heart_Data.pkl
